# BirdCLEF+ 2026 — EfficientNet Training (Kaggle GPU)

**Run on Kaggle with GPU accelerator enabled.**

- Model: EfficientNet-B0 (timm)
- Input: 128×160 mel spectrogram (5-second chunks)
- Augmentation: SpecAugment + Mixup
- Loss: BCEWithLogitsLoss (multi-label)
- CV: 5-fold stratified
- Optimizer: AdamW + CosineAnnealingLR

In [ ]:
# ── Check available input paths ───────────────────────────────────────
import os
print('=== /kaggle/input/ ===')
for d in sorted(os.listdir('/kaggle/input')):
    full = f'/kaggle/input/{d}'
    contents = os.listdir(full)[:5]
    print(f'  {d}/ -> {contents}')

In [ ]:
# ── Install deps ──────────────────────────────────────────────────────
!pip install timm librosa -q

In [ ]:
import os, sys, time, warnings, ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import librosa
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Paths: auto-detect competition data ───────────────────────────────
CANDIDATES = [
    '/kaggle/input/birdclef-2026',
    '/kaggle/input/birdclef2026',
]
COMP_DIR = next((p for p in CANDIDATES if os.path.exists(f'{p}/train.csv')), None)
assert COMP_DIR is not None, f'Competition data not found. Check /kaggle/input/ above.'
OUT_DIR = '/kaggle/working'
print(f'COMP_DIR: {COMP_DIR}')
print(f'Files: {os.listdir(COMP_DIR)[:8]}')

# ── Audio config ───────────────────────────────────────────────────────
SR       = 32000
DURATION = 5
N_FFT    = 1024
HOP_LEN  = 320
N_MELS   = 128
FMIN     = 20
FMAX     = 16000
IMG_H    = 128
IMG_W    = 160

# ── Training config ────────────────────────────────────────────────────
MODEL_NAME  = 'efficientnet_b0'
BATCH       = 32
LR          = 3e-4
EPOCHS      = 20
N_FOLDS     = 5
TRAIN_FOLDS = [0, 1, 2]
SEED        = 42
NUM_WORKERS = 2

np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────
train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
sub_df   = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')

species_list = [c for c in sub_df.columns if c != 'row_id']
label_to_idx = {s: i for i, s in enumerate(species_list)}

print(f'Train recordings: {len(train_df):,}')
print(f'Target species:   {len(species_list)}')

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────
def audio_to_melspec(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LEN,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_norm.astype(np.float32)

def load_random_chunk(path):
    try:
        total_dur = librosa.get_duration(path=path)
        if total_dur <= DURATION:
            audio, _ = librosa.load(path, sr=SR, mono=True)
        else:
            offset = np.random.uniform(0, total_dur - DURATION)
            audio, _ = librosa.load(path, sr=SR, mono=True,
                                    offset=offset, duration=DURATION)
        n = DURATION * SR
        return np.pad(audio, (0, max(0, n - len(audio))))[:n]
    except Exception:
        return np.zeros(DURATION * SR, dtype=np.float32)

def spec_augment(mel):
    mel = mel.copy()
    for _ in range(2):
        f = np.random.randint(0, 15)
        f0 = np.random.randint(0, N_MELS - f)
        mel[f0:f0+f, :] = 0.0
    _, T = mel.shape
    for _ in range(2):
        t = np.random.randint(0, 25)
        t0 = np.random.randint(0, max(1, T - t))
        mel[:, t0:t0+t] = 0.0
    return mel

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────
class BirdDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = f"{COMP_DIR}/train_audio/{row['filename']}"
        audio = load_random_chunk(path)
        mel   = audio_to_melspec(audio)

        if mel.shape[1] != IMG_W:
            mel = np.array([
                np.interp(np.linspace(0, mel.shape[1]-1, IMG_W),
                          np.arange(mel.shape[1]), mel[i])
                for i in range(mel.shape[0])
            ], dtype=np.float32)

        if self.augment:
            mel = spec_augment(mel)

        img = np.stack([mel, mel, mel], axis=0)  # (3, H, W)

        label = np.zeros(len(species_list), dtype=np.float32)
        primary = str(row['primary_label'])
        if primary in label_to_idx:
            label[label_to_idx[primary]] = 1.0
        try:
            for s in ast.literal_eval(str(row.get('secondary_labels', '[]'))):
                s = str(s)
                if s in label_to_idx:
                    label[label_to_idx[s]] = 0.5
        except Exception:
            pass

        return torch.from_numpy(img), torch.from_numpy(label)

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────
class BirdModel(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = timm.create_model(
            MODEL_NAME, pretrained=True, in_chans=3,
            num_classes=0, global_pool='avg'
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.num_features, n_classes)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    return lam * x + (1-lam) * x[idx], lam * y + (1-lam) * y[idx]

In [ ]:
# ── Training ───────────────────────────────────────────────────────────
def train_fold(fold, train_df, val_df):
    print(f'\n=== Fold {fold} | train={len(train_df)}, val={len(val_df)} ===')

    train_ds = BirdDataset(train_df, augment=True)
    val_ds   = BirdDataset(val_df,   augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    model     = BirdModel(n_classes=len(species_list)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=LR * 0.01)
    criterion = nn.BCEWithLogitsLoss()
    scaler    = GradScaler()

    best_auc, best_path = 0.0, f'{OUT_DIR}/model_fold{fold}.pt'

    for epoch in range(1, EPOCHS + 1):
        model.train()
        t_losses = []
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            imgs, labels = mixup(imgs, labels)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            t_losses.append(loss.item())
        scheduler.step()

        model.eval()
        all_preds, all_labels, v_losses = [], [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                with autocast():
                    logits = model(imgs)
                    v_losses.append(criterion(logits, labels).item())
                all_preds.append(torch.sigmoid(logits).cpu().numpy())
                all_labels.append(labels.cpu().numpy())

        preds  = np.vstack(all_preds)
        labels = np.vstack(all_labels)
        aucs   = [roc_auc_score(labels[:, i], preds[:, i])
                  for i in range(labels.shape[1]) if labels[:, i].sum() > 0]
        auc = np.mean(aucs) if aucs else 0.0

        print(f'  Ep{epoch:02d} train={np.mean(t_losses):.4f} '
              f'val={np.mean(v_losses):.4f} auc={auc:.4f}')

        if auc > best_auc:
            best_auc = auc
            torch.save({'state': model.state_dict(), 'auc': auc,
                        'species': species_list, 'model': MODEL_NAME}, best_path)
            print(f'    -> Saved (auc={auc:.4f})')

    print(f'Fold {fold} best AUC: {best_auc:.4f}')
    return best_auc


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(train_df, train_df['primary_label']))

fold_aucs = []
for fold in TRAIN_FOLDS:
    tr_idx, vl_idx = splits[fold]
    auc = train_fold(fold, train_df.iloc[tr_idx], train_df.iloc[vl_idx])
    fold_aucs.append(auc)

print(f'\nMean CV AUC: {np.mean(fold_aucs):.4f}')

In [ ]:
# List saved model weights
import glob
saved = glob.glob(f'{OUT_DIR}/model_fold*.pt')
for p in saved:
    print(f'{p}: {os.path.getsize(p)/1024/1024:.1f} MB')